# MwC 01 — setup and AAL3-144 recomputation

Recompute MwC across all 144 retained AAL3 nodes before extracting the formal 26 targets.

- Formal analysis universe: 12 cortical + 14 limbic ROIs (26 nodes)
- Cross-network connectivity: 12 × 14 = 168 edges
- Pipeline: `mwc`; step 1 of 5
- Run mode: standalone; all prerequisite steps load automatically in the same kernel.


In [1]:
from pathlib import Path
import os

_repo = Path(os.environ.get("METAFL_ROOT", Path.cwd())).resolve()
if not (_repo / "pipeline_steps").is_dir() and (_repo.parent / "pipeline_steps").is_dir():
    _repo = _repo.parent
if not (_repo / "pipeline_steps").is_dir():
    raise FileNotFoundError("Set METAFL_ROOT to the repository root")
os.environ["METAFL_ROOT"] = str(_repo)
os.chdir(_repo)

def _run_step(relative_path):
    _path = _repo / "pipeline_steps" / relative_path
    _code = compile(_path.read_text(encoding="utf-8"), str(_path), "exec")
    exec(_code, get_ipython().user_ns)


## Preamble


In [2]:
"""Auto-extracted pipeline step: MwC 01 — setup and AAL3-144 recomputation.

Do not edit generated copies independently of the source notebooks.
"""


'Auto-extracted pipeline step: MwC 01 — setup and AAL3-144 recomputation.\n\nDo not edit generated copies independently of the source notebooks.\n'

## Source 2


In [3]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import Image, Markdown, display
from matplotlib.patches import Patch
from scipy import stats

SEED = 20260903
PET_SCALE = 100.0
N_ROIS = 26
N_ANALYSIS_NODES = 144

BASE = Path(os.environ.get("METAFL_ROOT", Path.cwd())).resolve()
INPUT_ROOT = BASE / "inputs"
MWC_INPUT_ROOT = INPUT_ROOT / "mwc"
NORMATIVE_ROOT = MWC_INPUT_ROOT / "normative_sc"
OUTPUT_DIR = BASE / "outputs" / "mwc"
FIGURE_DIR = OUTPUT_DIR / "figures"
MWC_RECOMPUTED_ROOT = OUTPUT_DIR / "recomputed_normative_sc"
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
FIGURE_DIR.mkdir(parents=True)
os.environ.setdefault("MPLCONFIGDIR", str(BASE / ".matplotlib"))
np.random.seed(SEED)

MWC_CALCULATOR = BASE / "scripts" / "03_compute_normative_sc_mwc.py"
MWC_CALCULATION_INPUT_FILE = NORMATIVE_ROOT / "session_gcmi_pet_AAL3_144.npz"
MWC_MASK_FILE = NORMATIVE_ROOT / "normative_SC_top10_mask.npz"
MWC_REFERENCE_WIDE_FILE = NORMATIVE_ROOT / "subject_level_corticolimbic26_normativeSC10_MwC_wide.csv"
MWC_REFERENCE_FULL_FILE = NORMATIVE_ROOT / "subject_level_AAL3_144_normativeSC10_MwC_wide.csv.gz"
MWC_WIDE_FILE = MWC_RECOMPUTED_ROOT / "subject_level_corticolimbic26_normativeSC10_MwC_wide.csv"
MWC_FULL_FILE = MWC_RECOMPUTED_ROOT / "subject_level_AAL3_144_normativeSC10_MwC_wide.csv.gz"
MWC_CALCULATION_VALIDATION_FILE = MWC_RECOMPUTED_ROOT / "calculation_validation_report.json"
AAL_VALIDATION_FILE = NORMATIVE_ROOT / "validation_report.json"
ROI_FILE = INPUT_ROOT / "core" / "corticolimbic_12x14_ROI_definition.csv"
CLINICAL_FILE = INPUT_ROOT / "core" / "clinical_data.xlsx"
MWC_ADJUSTED_26_FILE = (
    NORMATIVE_ROOT / "age_sex_adjusted" /
    "subject_level_corticolimbic26_age_sex_adjusted_normativeSC10_MwC_wide.csv"
)
MWC_DEVIATION_HEATMAP = FIGURE_DIR / "Figure_nTRD_TRD1_TRD2_normativeSC_MwC_26ROI_deviation_heatmap.png"
MWC_SPATIAL_FIGURE = FIGURE_DIR / "Figure_TRD1_TRD2_normativeSC_MwC_26ROI_spatial_correlation.png"
MWC_SPATIAL_SUMMARY_FILE = OUTPUT_DIR / "TRD1_TRD2_normativeSC_MwC_26ROI_spatial_summary.csv"

required_files = [
    MWC_CALCULATOR,
    MWC_CALCULATION_INPUT_FILE,
    MWC_MASK_FILE, MWC_REFERENCE_WIDE_FILE, MWC_REFERENCE_FULL_FILE,
    AAL_VALIDATION_FILE, ROI_FILE, CLINICAL_FILE, MWC_ADJUSTED_26_FILE,
]
missing = [str(input_path) for input_path in required_files if not input_path.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))
display(pd.DataFrame({
    "input": [input_path.name for input_path in required_files],
    "size_kib": [round(input_path.stat().st_size / 1024, 1) for input_path in required_files],
}))


,input,size_kib
0,03_compute_normative_sc_mwc.py,12.4
1,session_gcmi_pet_AAL3_144.npz,17095.9
2,normative_SC_top10_mask.npz,53.7
3,subject_level_corticolimbic26_normativeSC10_Mw...,120.9
4,subject_level_AAL3_144_normativeSC10_MwC_wide....,288.0
5,validation_report.json,0.2
6,corticolimbic_12x14_ROI_definition.csv,0.6
7,clinical_data.xlsx,10.9
8,subject_level_corticolimbic26_age_sex_adjusted...,133.5


## Source 4


In [4]:
subprocess.run(
    [sys.executable, str(MWC_CALCULATOR), "--base", str(BASE)],
    check=True,
)
with MWC_CALCULATION_VALIDATION_FILE.open(encoding="utf-8") as stream:
    mwc_calculation_validation = json.load(stream)
assert mwc_calculation_validation["status"] == "PASS"
required_calculation_checks = [
    "sessions_253",
    "retained_AAL3_nodes_144",
    "fixed_dTOR_normative_SC_edges_1030",
    "gcmi_floor_bits_0p01",
    "row_normalization_error_below_1e_12",
    "all_144_nodes_computed_before_26_extraction",
    "isolated_nodes_remain_missing",
    "full_144_matches_validated_reference",
    "selected_26_matches_validated_reference",
]
assert all(mwc_calculation_validation["checks"][key] for key in required_calculation_checks)
display(pd.DataFrame([
    {"check": key, "passed": mwc_calculation_validation["checks"][key]}
    for key in required_calculation_checks
]))


{
  "full_144": "/Users/freddylee/Desktop/Coding_repository/Python_analysis/metabolism_TRD_release_refresh_20261007/outputs/mwc/recomputed_normative_sc/subject_level_AAL3_144_normativeSC10_MwC_wide.csv.gz",
  "selected_26": "/Users/freddylee/Desktop/Coding_repository/Python_analysis/metabolism_TRD_release_refresh_20261007/outputs/mwc/recomputed_normative_sc/subject_level_corticolimbic26_normativeSC10_MwC_wide.csv",
  "validation": "/Users/freddylee/Desktop/Coding_repository/Python_analysis/metabolism_TRD_release_refresh_20261007/outputs/mwc/recomputed_normative_sc/calculation_validation_report.json"
}
PASS: AAL3-144 normative-SC-constrained MwC recomputed before 26-ROI extraction


,check,passed
0,sessions_253,True
1,retained_AAL3_nodes_144,True
2,fixed_dTOR_normative_SC_edges_1030,True
3,gcmi_floor_bits_0p01,True
4,row_normalization_error_below_1e_12,True
5,all_144_nodes_computed_before_26_extraction,True
6,isolated_nodes_remain_missing,True
7,full_144_matches_validated_reference,True
8,selected_26_matches_validated_reference,True


## Results preview

Key numerical tables and figures generated by this step are displayed below. The full-resolution files remain under `outputs/`.


In [5]:
from io import BytesIO
import json as _preview_json
import pandas as _preview_pd
from PIL import Image as PILImage
from IPython.display import Image as NotebookImage, Markdown, display

_inline_table_specs = [{'title': 'MwC recomputation validation', 'path': 'outputs/mwc/recomputed_normative_sc/calculation_validation_report.json'}]
_inline_figure_paths = []

display(Markdown("### Key tables"))
for _table_spec in _inline_table_specs:
    _relative_table = _table_spec["path"]
    _table_path = _repo / _relative_table
    if not _table_path.is_file():
        raise FileNotFoundError(_table_path)
    if _table_path.suffix.lower() == ".csv":
        _preview_table = _preview_pd.read_csv(_table_path)
    elif _table_path.suffix.lower() in {".xlsx", ".xls"}:
        _preview_table = _preview_pd.read_excel(
            _table_path, sheet_name=_table_spec.get("sheet", 0)
        )
    elif _table_path.suffix.lower() == ".json":
        _payload = _preview_json.loads(_table_path.read_text(encoding="utf-8"))
        _checks = _payload.get("checks", _payload)
        _preview_table = _preview_pd.DataFrame(
            {"item": list(_checks), "value": list(_checks.values())}
        ) if isinstance(_checks, dict) else _preview_pd.json_normalize(_payload)
    else:
        raise ValueError(f"Unsupported table format: {_table_path.suffix}")
    if _table_spec.get("query"):
        _preview_table = _preview_table.query(_table_spec["query"]).reset_index(drop=True)
    _max_rows = int(_table_spec.get("max_rows", 40))
    display(Markdown(
        f"**{_table_spec['title']}**  \n"
        f"`{_relative_table}` — {len(_preview_table)} rows × {len(_preview_table.columns)} columns"
    ))
    with _preview_pd.option_context("display.max_columns", 40, "display.max_colwidth", 60):
        display(_preview_table.head(_max_rows))

if _inline_figure_paths:
    display(Markdown("### Figures"))
for _relative_figure in _inline_figure_paths:
    _figure_path = _repo / _relative_figure
    if not _figure_path.is_file():
        raise FileNotFoundError(_figure_path)
    with PILImage.open(_figure_path) as _source_image:
        _preview_image = _source_image.convert("RGB")
        _preview_image.thumbnail((1600, 1200), PILImage.Resampling.LANCZOS)
        _preview_buffer = BytesIO()
        _preview_image.save(_preview_buffer, format="PNG", optimize=True)
    display(Markdown(f"**`{_relative_figure}`**"))
    display(NotebookImage(data=_preview_buffer.getvalue(), format="png"))


### Key tables

**MwC recomputation validation**  
`outputs/mwc/recomputed_normative_sc/calculation_validation_report.json` — 12 rows × 2 columns

,item,value
0,sessions_253,True
1,retained_AAL3_nodes_144,True
2,possible_undirected_edges_10296,True
3,fixed_dTOR_normative_SC_edges_1030,True
4,structural_mask_symmetric,True
5,structural_mask_diagonal_zero,True
6,gcmi_floor_bits_0p01,True
7,row_normalization_error_below_1e_12,True
8,all_144_nodes_computed_before_26_extraction,True
9,isolated_nodes_remain_missing,True
